# MobilityDuck and DuckDB on the AIS data of the book (Chapter 7)

Every listing of Chapter 7 that reads the AIS data, in the order of the chapter, with the answer the chapter prints as a comment. Set before starting Jupyter:

- `AIS_DIR`: the directory of `aisdk-2026-01-20.csv` (`ais/fetch_data.sh day` of this repository downloads its archive);
- `PG_AIS`: the connection string of the AIS database built by `ais/build.sh` (for example `dbname=ais`);
- `MOBILITYDUCK_EXTENSION`: the path of `mobilityduck.duckdb_extension`.

The notebook runs DuckDB v1.5.5, as the README of this chapter explains.

In [1]:
import os

import duckdb
import pandas as pd

AIS_DIR = os.environ['AIS_DIR']
PG_AIS = os.environ['PG_AIS']
MOBILITYDUCK_EXTENSION = os.environ['MOBILITYDUCK_EXTENSION']
print(duckdb.__version__)

# The listings create their tables as the chapter prints them, so every run starts
# from a new database file
for f in ('ais_ch7.duckdb', 'ais_ch7.duckdb.wal'):
    if os.path.exists(f):
        os.remove(f)
con = duckdb.connect('ais_ch7.duckdb', config={'allow_unsigned_extensions': 'true'})
con.execute('INSTALL spatial FROM core_nightly')
con.execute('LOAD spatial')


def run(sql, rows=20):
    """Run the statements of a listing and show the first rows of the answer of the
    last one; only these rows are fetched, so that an answer of millions of rows, such
    as that of SELECT * EXCLUDE (SOG) FROM AISInput, stays in DuckDB."""
    sql = (sql.replace("'aisdk-2026-01-20.csv'", f"'{AIS_DIR}/aisdk-2026-01-20.csv'")
              .replace("'/path/to/mobilityduck.duckdb_extension'", f"'{MOBILITYDUCK_EXTENSION}'")
              .replace("'{conn_str}'", f"'{PG_AIS}'"))
    rel = con.execute(sql)
    if not rel.description:
        return None
    return pd.DataFrame(rel.fetchmany(rows), columns=[d[0] for d in rel.description])

1.5.5


## DuckDB: loading the AIS reports and SQL features

The listings of Sect. "DuckDB" on the raw AIS reports of January 20, 2026, loaded from the file `aisdk-2026-01-20.csv` of the Danish Maritime Authority (Appendix "The AIS Database of the Book").

In [2]:
run("""
SET geometry_always_xy = true;
CREATE TABLE AISInput AS
 SELECT *, ST_Point(Longitude, Latitude) AS Geom
 FROM read_csv('aisdk-2026-01-20.csv', header = true,
  timestampformat = '%d/%m/%Y %H:%M:%S',
  names = ['T', 'TypeOfMobile', 'MMSI', 'Latitude', 'Longitude',
    'NavigationalStatus', 'ROT', 'SOG', 'COG', 'Heading', 'IMO', 'CallSign',
    'Name', 'ShipType', 'CargoType', 'Width', 'Length',
    'TypeOfPositionFixingDevice', 'Draught', 'Destination', 'ETA',
    'DataSourceType', 'SizeA', 'SizeB', 'SizeC', 'SizeD']);
""")

,Count
0,17630815


In [3]:
run("""
SELECT * EXCLUDE (SOG)
FROM AISInput;
""")

,T,TypeOfMobile,MMSI,Latitude,Longitude,NavigationalStatus,ROT,COG,Heading,IMO,...,TypeOfPositionFixingDevice,Draught,Destination,ETA,DataSourceType,SizeA,SizeB,SizeC,SizeD,Geom
0,2026-01-20,Class A,246483000,56.245220,12.378778,Under way using engine,0.0,167.9,169.0,Unknown,...,Undefined,NaN,Unknown,NaT,AIS,NaN,NaN,NaN,NaN,b'\x01\x01\x00\x00\x00z\xe3\xa40\xef\xc1(@\x9e...
1,2026-01-20,Base Station,2190064,56.716515,11.519045,Unknown value,NaN,NaN,NaN,Unknown,...,GPS,NaN,Unknown,NaT,AIS,NaN,NaN,NaN,NaN,"b""\x01\x01\x00\x00\x00\xfdM(D\xc0\t'@\xf6\x0bv..."
2,2026-01-20,Class A,219031856,55.475085,8.421717,Under way using engine,0.0,267.0,242.0,Unknown,...,Undefined,NaN,Unknown,NaT,AIS,NaN,NaN,NaN,NaN,b'\x01\x01\x00\x00\x00\x9fUfJ\xeb\xd7 @\xfb\xe...
3,2026-01-20,Class A,246478000,57.422855,11.618723,Under way using engine,1.1,322.1,319.0,Unknown,...,Undefined,NaN,Unknown,NaT,AIS,NaN,NaN,NaN,NaN,"b""\x01\x01\x00\x00\x00\xe6\x90\xd4B\xc9<'@\x9c..."
4,2026-01-20,Class A,246483000,56.245220,12.378778,Under way using engine,0.0,167.9,169.0,Unknown,...,Undefined,NaN,Unknown,NaT,AIS,NaN,NaN,NaN,NaN,b'\x01\x01\x00\x00\x00z\xe3\xa40\xef\xc1(@\x9e...
5,2026-01-20,Class A,246478000,57.422855,11.618723,Under way using engine,1.1,322.1,319.0,Unknown,...,Undefined,NaN,Unknown,NaT,AIS,NaN,NaN,NaN,NaN,"b""\x01\x01\x00\x00\x00\xe6\x90\xd4B\xc9<'@\x9c..."
6,2026-01-20,Base Station,2194006,55.538870,5.033200,Unknown value,NaN,NaN,NaN,Unknown,...,Surveyed,NaN,Unknown,NaT,AIS,NaN,NaN,NaN,NaN,b'\x01\x01\x00\x00\x00\xa7\xe8H.\xff!\x14@\xd4...
7,2026-01-20,Class A,266464000,91.000000,0.000000,Unknown value,NaN,NaN,NaN,9816529,...,GPS,7.2,DKKAL,2026-01-20 23:00:00,AIS,94.0,26.0,3.0,16.0,b'\x01\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00...
8,2026-01-20,Class A,219005313,56.152330,10.251313,Under way using engine,NaN,301.3,NaN,Unknown,...,Undefined,NaN,Unknown,NaT,AIS,NaN,NaN,NaN,NaN,b'\x01\x01\x00\x00\x00\x8a\x1e\xf8\x18\xac\x80...
9,2026-01-20,Class A,273277840,54.940510,10.890440,Constrained by her draught,5.4,50.8,54.0,Unknown,...,Undefined,NaN,Unknown,NaT,AIS,NaN,NaN,NaN,NaN,b'\x01\x01\x00\x00\x00\xb9\x19n\xc0\xe7\xc7%@\...


In [4]:
run("""
SELECT MAX(COLUMNS('L.*')) FROM AISInput;
/* | Latitude | Longitude | Length |
   |  double  |  double   | int64  |
   |   91.0   | 40.337515 |   400  | */
""")

,Latitude,Longitude,Length
0,91.0,40.337515,400


In [5]:
run("""
SELECT SOG > 0.5 AS IsMoving, date_trunc('month', T) AS ReportMonth,
  AVG(ROT) AS AvgRot
FROM AISInput
WHERE IsMoving = TRUE
GROUP BY IsMoving, ReportMonth
HAVING AvgRot > 0.01;
/* | IsMoving | Month               | AvgRot |
   | true     | 2026-01-01 00:00:00 | 0.0129 | */
""")

,IsMoving,Month,AvgRot
0,True,2026-01-01,0.012871


In [6]:
run("""
SELECT MMSI, date_trunc('day', T) AS ReportDay, round(SUM(SOG) / 24, 2) 
  AS TotalSogPerDay, round(AVG(COG), 2) AS AvgCogPerDay
FROM AISInput
GROUP BY ALL;
/* | MMSI      | Day        | TotalSogPerDay | AvgCogPerDay |
   | 258656000 | 2026-01-20 |        3482.27 |       240.73 |
   ... */
""")

,MMSI,Day,TotalSogPerDay,AvgCogPerDay
0,244571000,2026-01-20,33.98,200.21
1,265031000,2026-01-20,10899.77,138.57
2,218098000,2026-01-20,8991.09,204.94
3,219007404,2026-01-20,0.09,241.75
4,2190075,2026-01-20,NaN,NaN
5,374135000,2026-01-20,6383.26,263.52
6,219024698,2026-01-20,1100.24,305.25
7,219023833,2026-01-20,4201.33,157.10
8,266365000,2026-01-20,0.25,NaN
9,219019017,2026-01-20,0.02,96.29


In [7]:
run("""
SELECT *
FROM (
  SELECT MMSI, date_trunc('day', T) AS ReportDay,  ShipType, SOG
  FROM AISInput
  WHERE MMSI = 258656000)
PIVOT (AVG(SOG) FOR ShipType IN ('Cargo', 'Tanker', 'Passenger', 'Fishing') )
ORDER BY ReportDay;
/* | MMSI      | day        | Cargo    | Tanker | Passenger | Fishing |
   | 258656000 | 2026-01-20 | 5.855474 | NULL   | 0.003226  | NULL    | */
""")

,MMSI,day,Cargo,Tanker,Passenger,Fishing
0,258656000,2026-01-20,5.855474,None,0.003226,None


In [8]:
run("""
SELECT *
FROM (
  SELECT MMSI, IF(hour(T) < 12, 'AM', 'PM') AS HalfDay, SOG
  FROM AISInput )
PIVOT (
  round(sum(SOG)) AS total,           -- total SOG per period
  max(SOG) AS bestSOG                 -- max SOG per period
  FOR HalfDay IN ('AM', 'PM')         -- pivot by half of the day
)
ORDER BY MMSI;
/* | MMSI    | AM_total | AM_bestSOG | PM_total | PM_bestSOG |
   | 3638    | 0.0      | 0.0        | 0.0      | 0.0        |
   | 261505  | NULL     | NULL       | 595.0    | 60.0       |
   | 2190045 | 1.0      | 0.1        | 0.0      | 0.0        |
   ... */
""")

,MMSI,AM_total,AM_bestSOG,PM_total,PM_bestSOG
0,3638,0.0,0.0,0.0,0.0
1,261505,NaN,NaN,595.0,60.0
2,2190045,1.0,0.1,0.0,0.0
3,2190047,NaN,NaN,NaN,NaN
4,2190048,NaN,NaN,NaN,NaN
5,2190049,NaN,NaN,NaN,NaN
6,2190051,NaN,NaN,NaN,NaN
7,2190052,NaN,NaN,NaN,NaN
8,2190064,NaN,NaN,NaN,NaN
9,2190065,NaN,NaN,NaN,NaN


## Querying DuckDB

The queries of Sect. "Querying DuckDB" on the table `AISInput`.

*Count the number of records in the input dataset.*

In [9]:
run("""
SELECT COUNT(*) FROM AISInput;
-- 17630815
""")

,count_star()
0,17630815


*Number of messages per vessel.*

In [10]:
run("""
SELECT MMSI, COUNT(*) AS MessageCount
FROM AISInput
GROUP BY MMSI
ORDER BY MessageCount DESC;
""")

,MMSI,MessageCount
0,2579999,101899
1,2190049,72461
2,2190068,61863
3,2190076,57390
4,2190048,53670
5,2190051,53658
6,3638,53416
7,219022903,53045
8,2190073,51655
9,2190067,51439


*First and last seen timestamps per vessel.*

In [11]:
run("""
SELECT MMSI, MIN(t) AS FirstSeen, MAX(t) AS LastSeen
FROM AISInput
GROUP BY MMSI
ORDER BY LastSeen;
""")

,MMSI,FirstSeen,LastSeen
0,246993000,2026-01-20 00:00:00,2026-01-20 00:00:00
1,246440000,2026-01-20 00:00:04,2026-01-20 00:00:34
2,305078000,2026-01-20 00:01:34,2026-01-20 00:01:34
3,211290200,2026-01-20 00:02:03,2026-01-20 00:04:31
4,257643000,2026-01-20 00:05:00,2026-01-20 00:05:00
5,314001119,2026-01-20 00:00:34,2026-01-20 00:10:53
6,210617000,2026-01-20 00:00:04,2026-01-20 00:11:40
7,44133113,2026-01-20 00:13:43,2026-01-20 00:13:43
8,992651036,2026-01-20 00:17:22,2026-01-20 00:17:22
9,255915667,2026-01-20 00:00:20,2026-01-20 00:18:03


*Average an maximum speed over ground (SOG) per vessel.*

In [12]:
run("""
SELECT  MMSI, AVG(SOG) AS AvgSpeed, MAX(SOG) AS MaxSpeed
FROM AISInput
GROUP BY MMSI
ORDER BY AvgSpeed DESC;
""")

,MMSI,AvgSpeed,MaxSpeed
0,226786696,212.000000,212.0
1,111257012,120.850000,150.0
2,111265555,116.584615,136.0
3,111211510,112.889916,140.0
4,111265581,103.000000,105.0
5,111219507,101.464695,102.2
6,111211512,99.639246,137.0
7,111261503,93.514825,122.0
8,111219513,91.562525,102.2
9,111219520,85.813205,102.2


*Active hours per vessel.*

In [13]:
run("""
SELECT MMSI, COUNT(DISTINCT DATE_TRUNC('hour', t)) AS ActiveHours
FROM AISInput
GROUP BY MMSI
ORDER BY ActiveHours DESC;
""")

,MMSI,ActiveHours
0,219024402,24
1,253213000,24
2,219000781,24
3,212797000,24
4,219004529,24
5,314844000,24
6,220242000,24
7,219005511,24
8,219000775,24
9,255806478,24


*Hourly centroid of vessel movement.*

In [14]:
run("""
SELECT MMSI, DATE_TRUNC('hour', t) AS ReportHour,
  ST_Centroid(ST_Collect(ARRAY_AGG(geom))) AS Centroid
FROM AISInput
GROUP BY MMSI, ReportHour
ORDER BY MMSI, ReportHour;
""")

,MMSI,Hour,Centroid
0,3638,2026-01-20 00:00:00,"b'\x01\x01\x00\x00\x00\x11SR\x13\x03\xee%@Q,\x..."
1,3638,2026-01-20 01:00:00,b'\x01\x01\x00\x00\x00\xce~\xc0\xdb\x03\xee%@\...
2,3638,2026-01-20 02:00:00,"b""\x01\x01\x00\x00\x00\xe3\xa90&\x04\xee%@\xb3..."
3,3638,2026-01-20 03:00:00,b'\x01\x01\x00\x00\x00\xf5Q}^\x02\xee%@\xa4\xc...
4,3638,2026-01-20 04:00:00,b'\x01\x01\x00\x00\x00\x1d\xdc\xc5U\x02\xee%@\...
5,3638,2026-01-20 05:00:00,"b'\x01\x01\x00\x00\x00\xd1o\xd2\xea\x01\xee%@,..."
6,3638,2026-01-20 06:00:00,b'\x01\x01\x00\x00\x00\xff\xf0\xe8-\x02\xee%@\...
7,3638,2026-01-20 07:00:00,b'\x01\x01\x00\x00\x00)O\xcd~\x02\xee%@t\xb8\x...
8,3638,2026-01-20 08:00:00,b'\x01\x01\x00\x00\x00f\xf3<e\x00\xee%@\xe78n\...
9,3638,2026-01-20 09:00:00,b'\x01\x01\x00\x00\x00\xcb\xf9\x17X\xfc\xed%@\...


*Bounding box of all AIS positions.*

In [15]:
run("""
SELECT MIN(Latitude) AS MinLat, MAX(Latitude) AS MaxLat,
  MIN(Longitude) AS MinLon, MAX(Longitude) AS MaxLon
FROM AISInput;
""")

,MinLat,MaxLat,MinLon,MaxLon
0,-83.4967,91.0,-115.871855,40.337515


*Ships within 1,000 m of a given ship at a given time, with a margin of one minute.*

In [16]:
run("""
 WITH Target AS (
  SELECT Geom, t
  FROM AISInput
  WHERE MMSI = 219015442
  ORDER BY t DESC
  LIMIT 1 )
SELECT other.MMSI, other.t, ST_AsText(other.Geom) AS Geom
FROM AISInput other, Target t
WHERE other.MMSI <> 219015442 AND
  ST_Distance_Sphere(other.Geom, t.Geom) <= 1000 AND
  ABS(EXTRACT(EPOCH FROM (other.t - t.t))) <= 60;
""")

,MMSI,T,Geom
0,219007027,2026-01-20 23:58:39,POINT (10.585922 57.715918)
1,219007027,2026-01-20 23:58:39,POINT (10.585922 57.715918)
2,219020150,2026-01-20 23:58:39,POINT (10.58757 57.715627)
3,219005732,2026-01-20 23:58:40,POINT (10.586303 57.717898)
4,259414000,2026-01-20 23:58:40,POINT (10.592792 57.71838)
5,251323000,2026-01-20 23:58:40,POINT (10.587627 57.716725)
6,265019000,2026-01-20 23:58:40,POINT (10.600387 57.719012)
7,220141000,2026-01-20 23:58:40,POINT (10.595192 57.71787)
8,220141000,2026-01-20 23:58:40,POINT (10.595192 57.71787)
9,220076700,2026-01-20 23:58:41,POINT (10.587072 57.714893)


*For every ship, count the number of neighbors within 1,000 m considering the ship's latest position (consider space and time, with a margin of one minute).*

In [17]:
run("""
WITH Latest AS (
  SELECT MMSI, geom, t
  FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY MMSI
      ORDER BY t DESC) AS RowNo
    FROM AISInput )
  WHERE RowNo = 1 ),
Neighbors AS (
  SELECT l1.MMSI AS MMSI1, l2.MMSI AS MMSI2
  FROM Latest l1, Latest l2
  WHERE l1.MMSI <> l2.MMSI AND
    ST_Distance_Sphere(l1.Geom, l2.Geom) <= 1000 AND
    ABS(EXTRACT(EPOCH FROM (l1.t - l2.t))) <= 60 )
SELECT l.MMSI, ST_AsText(l.Geom) AS Geom, COUNT(n.MMSI2) AS NeighborCount
FROM Latest l LEFT JOIN Neighbors n ON l.MMSI = n.MMSI1
GROUP BY l.MMSI, l.Geom
ORDER BY NeighborCount DESC;
""")

,MMSI,Geom,NeighborCount
0,219007727,POINT (9.962617 57.592465),48
1,219008783,POINT (9.961003 57.59277),47
2,219001111,POINT (10.591937 57.718913),47
3,219007471,POINT (9.964123 57.592407),47
4,219014692,POINT (10.586925 57.715635),47
5,219005803,POINT (9.96216 57.592787),47
6,219015595,POINT (9.95605 57.591872),46
7,219031429,POINT (9.963523 57.592705),45
8,219027752,POINT (9.964912 57.592912),45
9,219015442,POINT (10.58772 57.716918),45


*Five nearest neighbors of our target ship considering space and time.*

In [18]:
run("""
WITH Latest AS (
  SELECT MMSI, geom, t
  FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY MMSI
      ORDER BY t DESC) AS RowNo
    FROM AISInput )
  WHERE RowNo = 1 ),
Target AS (
  SELECT MMSI, geom, t
  FROM Latest
  WHERE MMSI = 219015442 )
SELECT o.MMSI, ST_AsText(o.geom) AS geom,
  ST_Distance_Sphere(o.geom, t.geom) AS Distance,
  ABS(EXTRACT(EPOCH FROM (o.t - t.t))) AS TimeDiff
FROM Latest o JOIN Target t ON o.MMSI <> t.MMSI
WHERE ST_Distance_Sphere(o.geom, t.geom) <= 1000 AND
  ABS(EXTRACT(EPOCH FROM (o.t - t.t))) <= 60
ORDER BY Distance
LIMIT 5;

/* | MMSI | Geom | Distance | TimeDiff |
   | 251323000 | POINT (10.587627 57.716725) |  22.16 | 59.0 |
   ...
*/
""")

,MMSI,geom,Distance,TimeDiff
0,251323000,POINT (10.587627 57.716725),22.159973,59.0
1,219002827,POINT (10.587328 57.71746),64.607833,18.0
2,220343000,POINT (10.586863 57.717345),69.604797,19.0
3,220333000,POINT (10.587002 57.717613),88.264086,3.0
4,219034309,POINT (10.586427 57.716427),94.221460,18.0


## MobilityDuck: AIS trajectories

The queries of Sect. "Querying MobilityDuck" on the table `Ships` of the AIS database of the book, read from PostgreSQL with `postgres_scan`.

In [19]:
run("""
INSTALL spatial FROM core_nightly;
INSTALL '/path/to/mobilityduck.duckdb_extension';
LOAD spatial;
LOAD mobilityduck;

CREATE OR REPLACE TABLE ships (MMSI integer, Trip tgeompoint, SOG tfloat, 
 COG tfloat);

INSERT INTO ships
SELECT MMSI, Trip, SOG, COG
FROM postgres_scan('{conn_str}', 'public', 'ships');

SET threads=1;
SET memory_limit='7000MB';

SELECT COUNT(*) FROM ships;
-- 2701

ALTER TABLE ships ADD column trajectory geometry;
UPDATE ships set trajectory = trajectory(trip);
CREATE INDEX rtree_trip ON Ships USING RTREE(trajectory);
""")

,Count


*Number of ships at each hour between 1pm and 7pm.*

In [20]:
run("""
WITH Hours AS (
  SELECT TIMESTAMPTZ '2026-01-20 13:00:00+00' AS hour_start
  UNION ALL SELECT TIMESTAMPTZ '2026-01-20 14:00:00+00'
  UNION ALL SELECT TIMESTAMPTZ '2026-01-20 15:00:00+00'
  UNION ALL SELECT TIMESTAMPTZ '2026-01-20 16:00:00+00'
  UNION ALL SELECT TIMESTAMPTZ '2026-01-20 17:00:00+00'
  UNION ALL SELECT TIMESTAMPTZ '2026-01-20 18:00:00+00'
  UNION ALL SELECT TIMESTAMPTZ '2026-01-20 19:00:00+00' )
SELECT h.hour_start, COUNT(DISTINCT s.MMSI) AS num_ships
FROM Hours h, Ships s
WHERE atTime(s.Trip, h.hour_start) IS NOT NULL
GROUP BY h.hour_start
ORDER BY h.hour_start;
/* hour_start               num_ships
   2026-01-20 14:00:00+01        1657
   2026-01-20 15:00:00+01        1690
   2026-01-20 16:00:00+01        1657
   2026-01-20 17:00:00+01        1650
   2026-01-20 18:00:00+01        1690
   2026-01-20 19:00:00+01        1711
   2026-01-20 20:00:00+01        1692 */
""")

,hour_start,num_ships
0,2026-01-20 14:00:00+01:00,1657
1,2026-01-20 15:00:00+01:00,1690
2,2026-01-20 16:00:00+01:00,1657
3,2026-01-20 17:00:00+01:00,1650
4,2026-01-20 18:00:00+01:00,1690
5,2026-01-20 19:00:00+01:00,1711
6,2026-01-20 20:00:00+01:00,1692


*Ships within the port of Göteborg.*

In [21]:
run("""
WITH Goteborg(GotEnv) AS (
  SELECT ST_MakeEnvelope(666538, 6392057, 679171, 6403745))
SELECT s.MMSI,
ST_AsText(trajectory(atGeometry(s.Trip,  g.GotEnv))) as traj, g.GotEnv
FROM Goteborg g, Ships s
WHERE eIntersects(s.Trip, g.GotEnv);
""")

,MMSI,traj,GotEnv
0,211839040,MULTILINESTRING ((678604.1785360001 6403536.86...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...
1,219003122,GEOMETRYCOLLECTION (MULTIPOINT (675909.8930258...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...
2,219026170,LINESTRING (670590.3506348464 6392778.33383487...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...
3,219029880,GEOMETRYCOLLECTION (POINT (671572.7918987392 6...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...
4,219034887,"LINESTRING (670588.79314454 6392839.351280836,...",b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...
5,219168000,LINESTRING (670603.0223162144 6392778.31097663...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...
6,219437000,LINESTRING (669434.9280250781 6397817.33107010...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...
7,265012090,GEOMETRYCOLLECTION (MULTIPOINT (677244.3666781...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...
8,265026000,LINESTRING (670592.7399401724 6392735.85476075...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...
9,265044670,GEOMETRYCOLLECTION (MULTIPOINT (672364.6386011...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...


*Compute and list the vessel's trajectories between the ports of R{\o*

In [22]:
run("""
WITH Ports(Rodby, Puttgarden) AS
 (SELECT ST_MakeEnvelope(651135, 6058230, 651422, 6058548),
     ST_MakeEnvelope(644339, 6042108, 644896, 6042487))
SELECT s.MMSI, ST_AsText(trajectory(trip)) AS Trajectory,
 Rodby, Puttgarden
FROM Ports p, Ships s
WHERE eintersects(trip, p.Rodby) AND eintersects(trip, p.Puttgarden);
""")

,MMSI,Trajectory,Rodby,Puttgarden
0,211190000,LINESTRING (646261.7990774247 6046358.79709851...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...
1,219000429,LINESTRING (650530.3215489659 6057549.03231307...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...
2,211757540,LINESTRING (644418.9075008206 6041768.43210913...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...
3,219000431,LINESTRING (644884.6442705076 6044842.92779543...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...


*Compute and display the vessel activity between the ports of R{\o*

In [23]:
run("""
WITH AlertBelt(Belt) AS (
SELECT ST_MakeEnvelope(640730, 6058230, 654100, 6042487))
SELECT MMSI, atGeometry(s.Trip, b.Belt),
 ST_AsText(trajectory(atGeometry(s.Trip,  b.Belt))) as traj
FROM ships s, AlertBelt b
WHERE eintersects(s.Trip, b.Belt)
""")

,MMSI,"atgeometry(s.Trip, b.Belt)",traj
0,244750806,b'\x80\xfe\x15\x00.\x03^\x00\x07\x00\x00\x00\x...,MULTILINESTRING ((653505.8648616202 6051870.88...
1,244770273,b'@\xeb\x00\x00.\x03^\x00\x01\x00\x00\x00\t\x0...,LINESTRING (654100.0000001129 6046910.08507629...
2,244780000,b'\xc0\xf5\x00\x00.\x03^\x00\x01\x00\x00\x00\x...,LINESTRING (640729.9999995576 6049590.55406683...
3,244995000,b'`\xf8\x00\x00.\x03^\x00\x01\x00\x00\x00\x18\...,LINESTRING (640729.9999954373 6050614.33844982...
4,244997000,b'`\xff\x00\x00.\x03^\x00\x01\x00\x00\x00 \x01...,LINESTRING (654100.0000045625 6047088.00222166...
5,245043000,b'\x80r#\x00.\x03^\x00\x01\x00\x00\x00\x7f(\x0...,LINESTRING (651560.3173367763 6056516.14944588...
6,245258000,b'@t\x00\x00.\x03^\x00\x01\x00\x00\x00\x81\x00...,LINESTRING (640729.999998746 6050185.031541792...
7,245313000,b'@\x00\x01\x00.\x03^\x00\x01\x00\x00\x00!\x01...,LINESTRING (654100.0000028458 6047220.65950961...
8,245319000,b'\x00\xe7\x0c\x00.\x03^\x00\x01\x00\x00\x00\x...,LINESTRING (653190.9753798562 6057114.22305234...
9,245489000,b' 1\x1f\x00.\x03^\x00\x02\x00\x00\x00\xa0#\x0...,MULTILINESTRING ((653196.4821247084 6057064.07...


*Compute and display the vessels that during a time interval have been at 300m from each other, displaying the points, the closest distance, and the position of the vessels.*

In [24]:
run("""
WITH Timespan(Period) AS (
 SELECT tstzspan '[2026-01-20 11:00:00+00, 2026-01-20 23:30:00+00)' ),
TimeShips(MMSI, Trip) AS (
 SELECT MMSI, atTime(s.Trip, t.Period)
 FROM Ships s TABLESAMPLE RESERVOIR (300 ROWS) REPEATABLE (1),
   Timespan t
 WHERE atTime(s.Trip, t.Period) IS NOT NULL ),
TimeClosestShips(MMSI1, MMSI2, ClosestDist, TimeAtClosestDist, Trip1, Trip2) AS (
 SELECT s1.MMSI, s2.MMSI, startValue(atMin(s1.Trip <-> s2.Trip)),
  startTimestamp(atMin(s1.Trip <-> s2.Trip)), s1.Trip, s2.Trip
 FROM TimeShips s1, TimeShips s2
 WHERE s1.MMSI < s2.MMSI AND CASE WHEN eDwithin(s1.Trip, s2.Trip, 300)
  THEN startValue(atMin(s1.Trip <-> s2.Trip)) < 300 END )
SELECT t.MMSI1, t.MMSI2, t.ClosestDist, t.TimeAtClosestDist,
 valueAtTimestamp(Trip1, TimeAtClosestDist) AS Point1,
 valueAtTimestamp(Trip2, TimeAtClosestDist) AS Point2
FROM TimeClosestShips t;
""")

,MMSI1,MMSI2,ClosestDist,TimeAtClosestDist,Point1,Point2
0,211222800,211233510,251.091628,2026-01-20 21:18:46+01:00,b'\x01\x01\x00\x00\x00k\x8e_x\xa2]\x1dA\x91\xc...,b'\x01\x01\x00\x00\x00N\x82\xa2m4Z\x1dA\xa0[g\...
1,205597000,211448440,91.065605,2026-01-20 20:51:37.901535+01:00,b'\x01\x01\x00\x00\x00\x90\x13\xc3\x93\xf9a\x1...,"b""\x01\x01\x00\x00\x00\xe9Ge\xd1\xfdb\x1dAq'\x..."
2,211222800,211448440,277.731411,2026-01-20 21:18:46+01:00,b'\x01\x01\x00\x00\x00k\x8e_x\xa2]\x1dA\x91\xc...,b'\x01\x01\x00\x00\x00a\xee\xc2\xbcKY\x1dA$\x8...
3,211233510,211448440,136.753248,2026-01-20 21:18:30+01:00,b'\x01\x01\x00\x00\x00a\xf5\xe2\xc7*Z\x1dA\xf5...,b'\x01\x01\x00\x00\x00\xc5O\xd1EmY\x1dA\x83N\x...
4,211369510,211727510,22.981915,2026-01-21 00:25:52+01:00,b'\x01\x01\x00\x00\x00\xf0s\xa9\xd5\xb9n%A\x10...,b'\x01\x01\x00\x00\x00\x83*IX\xc1n%AGB\xe6E\t\...
5,211727510,211787790,75.746467,2026-01-20 20:33:59+01:00,b'\x01\x01\x00\x00\x00\xbe\xa7Y\xa5\xc8l%A\x1d...,b'\x01\x01\x00\x00\x00\xf2#\xe7\xfbXm%A\x07\x9...
6,211450850,218762000,168.580037,2026-01-20 13:29:29+01:00,b'\x01\x01\x00\x00\x00S\x9a\xef\xf4>_\x1dA\xcb...,b'\x01\x01\x00\x00\x00\xe6\xef\xc2\xfc\xcfa\x1...
7,219000416,219000527,88.213469,2026-01-20 14:11:15+01:00,"b'\x01\x01\x00\x00\x00N{-\xb0\x98\x13""A\xfc\xf...","b'\x01\x01\x00\x00\x00+\xdc\x1f\xe3\x0c\x13""A\..."
8,209702000,219001275,26.545043,2026-01-20 14:25:30+01:00,b'\x01\x01\x00\x00\x00T*\x12\xb9q\xd8\x1cA\xe2...,b'\x01\x01\x00\x00\x00\xcc\xad\x07l*\xd8\x1cA\...
9,219002838,219007477,75.867063,2026-01-20 16:26:06+01:00,b'\x01\x01\x00\x00\x0068\xa8\xdabK\x1cAI`\xc2\...,b'\x01\x01\x00\x00\x00E_!P\x92L\x1cA+0\xec\xf0...


*Compute and display the trajectories*

In [25]:
run("""
WITH Timespan(Period) AS (
 SELECT tstzspan '[2026-01-20 11:00:00+00, 2026-01-20 23:30:00+00)' ),
TimeShips(MMSI, Trip) AS (
 SELECT MMSI, atTime(s.Trip, t.Period)
 FROM Ships s TABLESAMPLE RESERVOIR (300 ROWS) REPEATABLE (1),
   Timespan t
 WHERE atTime(s.Trip, t.Period) IS NOT NULL )
SELECT s1.MMSI, s2.MMSI, s1.Trip, s2.Trip,
 ST_AsText(trajectory(s1.Trip)) as traj1,
 ST_AsText(trajectory(s2.Trip)) as traj2
FROM TimeShips s1, TimeShips s2
WHERE s1.MMSI < s2.MMSI AND eDwithin(s1.Trip, s2.Trip, 300)  ;
""")

,MMSI,MMSI,Trip,Trip,traj1,traj2
0,211222800,211233510,b' \x0c\x00\x00.\x03^\x00\x04\x00\x00\x00\x04\...,b'\xc0\xcb\x00\x00.\x03^\x00\x0e\x00\x00\x00\x...,MULTIPOINT (481135.39071816887 5968350.4438201...,GEOMETRYCOLLECTION (MULTIPOINT (480906.0175303...
1,205597000,211448440,b'`\x84\x01\x00.\x03^\x00\x10\x00\x00\x00\x9a\...,b'\xe0\xcc\x03\x00.\x03^\x00\x08\x00\x00\x00F\...,GEOMETRYCOLLECTION (MULTIPOINT (481473.4358787...,MULTILINESTRING ((481017.0396857883 5969073.42...
2,211222800,211448440,b' \x0c\x00\x00.\x03^\x00\x04\x00\x00\x00\x04\...,b'\xe0\xcc\x03\x00.\x03^\x00\x08\x00\x00\x00F\...,MULTIPOINT (481135.39071816887 5968350.4438201...,MULTILINESTRING ((481017.0396857883 5969073.42...
3,211233510,211448440,b'\xc0\xcb\x00\x00.\x03^\x00\x0e\x00\x00\x00\x...,b'\xe0\xcc\x03\x00.\x03^\x00\x08\x00\x00\x00F\...,GEOMETRYCOLLECTION (MULTIPOINT (480906.0175303...,MULTILINESTRING ((481017.0396857883 5969073.42...
4,211369510,211727510,b'\x00\x14\x00\x00.\x03^\x00\x01\x00\x00\x00\x...,b' w\x11\x00.\x03^\x00\x01\x00\x00\x00\xf2\x13...,LINESTRING (701121.9662196543 6009672.06127156...,LINESTRING (697127.9079792672 6021635.31775669...
5,211727510,211787790,b' w\x11\x00.\x03^\x00\x01\x00\x00\x00\xf2\x13...,b'\xe0\x0c\x02\x00.\x03^\x00\x08\x00\x00\x00F\...,LINESTRING (697127.9079792672 6021635.31775669...,MULTILINESTRING ((702119.3632266868 6007586.43...
6,211450850,218762000,b' \x13\x00\x00.\x03^\x00\x06\x00\x00\x00\x08\...,b'@t\x00\x00.\x03^\x00\r\x00\x00\x00i\x00\x00\...,GEOMETRYCOLLECTION (MULTIPOINT (481231.7391952...,GEOMETRYCOLLECTION (MULTIPOINT (481395.7107540...
7,219000416,219000527,b'`)\x01\x00.\x03^\x00\x01\x00\x00\x00P\x01\x0...,b' \x05\x00\x00.\x03^\x00\x01\x00\x00\x00\x02\...,LINESTRING (592333.4000197831 6366444.66749038...,POINT (592262.4436024477 6366385.872972894)
8,209702000,219001275,b'`E\x01\x00.\x03^\x00\x01\x00\x00\x00p\x01\x0...,b'\x00 \x0b\x00.\x03^\x00\x02\x00\x00\x00\xb1\...,LINESTRING (472602.0348101792 6104584.17999231...,MULTILINESTRING ((472258.828071299 6104575.734...
9,219002838,219007477,b'\x00\x02\x0f\x00.\x03^\x00\x01\x00\x00\x00#\...,b'\xe0\x1a\x02\x00.\x03^\x00\x01\x00\x00\x00d\...,LINESTRING (463567.0712436036 6147620.15503956...,LINESTRING (463655.73050722975 6147609.4985651...


*Trajectories passing through a buffer with a radius of 500m around the port of Copenhagen.*

In [26]:
run("""
WITH Port AS (
  SELECT ST_Point(726765.3929300838, 6178705.65508127) AS port_geom ),
PortBuffer AS (
  SELECT ST_Buffer(port_geom, 500) AS buf
  FROM Port )
SELECT s.MMSI, ST_AsText(trajectory(s.Trip)) AS trip_wkt,
    ST_AsWKB(pb.buf) AS buffer_wkb
FROM Ships s, PortBuffer pb
WHERE eIntersects(s.Trip, pb.buf);
""")

,MMSI,trip_wkt,buffer_wkb
0,219024675,LINESTRING (732793.6246569207 6165408.62672716...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0...
1,219002798,LINESTRING (727707.1985358351 6177304.80292357...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0...
2,219385000,LINESTRING (725887.4127071316 6176231.61954977...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0...


*Trajectories that stayed within radius of 1000m around the port of Copenhagen for at least twenty minutes.*

In [27]:
run("""
WITH Port AS (
  SELECT ST_Point(726765.3929300838, 6178705.65508127) AS port_geom ),
PortBuffer AS (
  SELECT ST_Buffer(port_geom, 1000) AS buf
  FROM Port )
SELECT s.MMSI,
   asText(stops(atGeometry(s.trip, pb.buf),10.0, interval '20 minutes')),
   ST_AsText(trajectory(atGeometry(s.trip, pb.buf)))
   AS tripWkt, ST_AsWKB(pb.buf) AS bufferWkb
FROM ships s, PortBuffer pb
WHERE eIntersects(s.trip, pb.buf) AND
 astext(stops(atGeometry(s.trip, pb.buf),10.0, interval '20 minutes'))
 IS NOT NULL;
""")

,MMSI,"astext(stops(atgeometry(s.trip, pb.buf), 10.0, CAST('20 minutes' AS INTERVAL)))",tripWkt,bufferWkb
0,219025528,{[POINT(726928.18481275 6179415.560635618)@202...,MULTILINESTRING ((727131.6033996923 6179634.53...,"b""\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0..."
1,219002798,{[POINT(727336.4119952423 6179379.998396069)@2...,MULTILINESTRING ((727130.8289535276 6179634.76...,"b""\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0..."
2,219003121,{[POINT(727372.28430405 6179375.516463043)@202...,LINESTRING (727374.782006778 6179372.301548108...,"b""\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0..."
3,219031423,{[POINT(726251.2063936263 6179537.872890741)@2...,LINESTRING (726251.2063936263 6179537.87289074...,"b""\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0..."
4,219001362,{[POINT(726975.5530205681 6179514.156220158)@2...,MULTILINESTRING ((727091.316211704 6179646.752...,"b""\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0..."
5,219002801,{[POINT(726928.247745734 6179491.616234032)@20...,MULTILINESTRING ((727135.1340101883 6179633.46...,"b""\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0..."


*Trajectories within a radius of 1000m around the port of Malmoö, for at least one hour. In this case, consider the port of Malmö as an area with a radius of 5000m from the point with coordinates (752775.913, 6167956.005, SRID = 25832). Also compute the start instant of this stop.*

In [28]:
run("""
WITH Port AS (
  SELECT ST_Point(752775.9137447166, 6167956.005944439) AS port_geom ),
PortBuffer AS (
  SELECT ST_Buffer(port_geom, 5000) AS buf
  FROM Port )
SELECT s.MMSI,
  asText(stops(atGeometry(s.trip, pb.buf),1000.0, interval '1 hour')),
  ST_AsText(trajectory(atGeometry(s.trip,   pb.buf)))
  AS tripWkt, ST_AsWKB(pb.buf) AS buffer_wkb,
  startTimestamp(stops(atGeometry(s.trip, pb.buf),1000.0,
  interval '1 hour')) AS start_time
FROM Ships s, PortBuffer pb
WHERE eIntersects(s.trip, pb.buf) AND
  astext(stops(atGeometry(s.trip, pb.buf),1000.0, interval '1 hour'))
  IS NOT NULL;
""")

,MMSI,"astext(stops(atgeometry(s.trip, pb.buf), 1000.0, CAST('1 hour' AS INTERVAL)))",tripWkt,buffer_wkb,start_time
0,212797000,{[POINT(752635.6332655648 6172942.189523194)@2...,MULTILINESTRING ((752639.0154889377 6172940.01...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0...,2026-01-20 18:23:53.108025+01:00
1,219159000,{[POINT(751233.5084446241 6170938.06546821)@20...,MULTILINESTRING ((750167.4360563783 6172203.88...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0...,2026-01-20 04:27:53+01:00
2,232059804,{[POINT(753007.2855194729 6171611.757306266)@2...,LINESTRING (750139.2598362201 6172188.82551294...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0...,2026-01-20 08:24:59+01:00
3,265514800,{[POINT(751714.0670077421 6170332.867291322)@2...,LINESTRING (751714.0670077421 6170332.86729132...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0...,2026-01-20 01:00:04+01:00
4,265594380,{[POINT(753936.2420363978 6171684.916205654)@2...,LINESTRING (753936.2420363978 6171684.91620565...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0...,2026-01-20 10:20:38+01:00
5,265607640,{[POINT(753976.3883683694 6171675.203821069)@2...,LINESTRING (753976.3883683694 6171675.20382106...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0...,2026-01-20 01:01:16+01:00
6,265610950,{[POINT(750924.7417881248 6170938.931090504)@2...,MULTILINESTRING ((751705.7607960323 6170343.32...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0...,2026-01-20 02:51:59+01:00
7,265757700,{[POINT(754020.8619551279 6171697.428819366)@2...,LINESTRING (754020.8619551279 6171697.42881936...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0...,2026-01-20 01:00:03+01:00
8,265821600,{[POINT(753995.2251142988 6171687.791822463)@2...,MULTILINESTRING ((753995.2251142988 6171687.79...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0...,2026-01-20 01:00:05+01:00
9,219027899,{[POINT(751211.0599393302 6170907.990429523)@2...,LINESTRING (750235.3507725103 6172240.18719179...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0...,2026-01-20 16:45:57+01:00
